# The 16MB Constraint

Where the bytes go, how quantization and compression fit the model under the limit, and the tradeoffs competitors are making.

## Where the Bytes Come From

The model's parameters are just numbers stored in memory. Every number takes up space. The question is: how many numbers do you have, and how many bytes does each one cost?

The baseline has ~25 million parameters. These are the numbers inside all the weight matrices:
- **Per attention layer (×9 blocks):** Q projection (512×512 = 262K), K projection (512×256 = 131K), V projection (512×256 = 131K), output projection (512×512 = 262K)
- **Per MLP layer (×9 blocks):** up-projection (512×1024 = 524K), down-projection (1024×512 = 524K)
- **Token embedding:** 1024 vocab × 512 dims = 524K (also the output layer, via tied embeddings)
- **Small stuff:** `attn_scale`, `mlp_scale`, `resid_mix`, `q_gain`, `skip_weights`, norms — a few thousand numbers total

During training, each of those numbers is stored as a 32-bit float — 4 bytes per number. ~25 million × 4 bytes = ~100MB. The competition limit is 16MB.

What counts toward 16MB: the compressed model bytes **plus** the code bytes in `train_gpt.py`. In practice the code is tiny compared to the model, but it's technically part of the budget.

## Int8 Quantization

Instead of storing each number as a 32-bit float (4 bytes, very precise), convert it to an 8-bit integer (1 byte, much less precise). 4× size reduction.

The way it works, for each row of a weight matrix:
1. Find the largest absolute value in the row. That becomes the **scale**.
2. Divide everything in the row by the scale, so all values fall between -1 and 1.
3. Multiply by 127 and round to the nearest integer. Now everything is between -127 and 127.
4. Store that integer in 1 byte. Store the scale separately (one float per row).

To reconstruct later: take the integer, divide by 127, multiply by the scale. You get back an approximation of the original float. The rounding introduces error, but it's surprisingly small — most of the model's behavior is preserved.

Think of it like rounding 3.14159 to 3.14. You lost some information, but for most purposes the answer is close enough. Quantization does this systematically across every number in the model.

Gets ~100MB down to ~25MB. Closer to the limit, but still over.

## Zlib Compression

After quantizing to int8, many of the integer values are similar or follow patterns — nearby weights in the same row tend to have similar magnitudes. Standard compression (same family as zip) exploits that redundancy. It finds repeated patterns and encodes them more efficiently, like how "aaaaaaa" can be stored as "7×a" instead of writing out all seven characters.

The quantized model gets zlib-compressed, and that compressed size is what counts toward the 16MB limit. This is what gets the ~25MB of int8 data under the wire.

## Baseline Size Breakdown

| Component | Numbers | Storage | Size |
|---|---|---|---|
| Token embedding (tied) | 1024 × 512 = 524,288 | FP16 (2 bytes) | ~1 MB |
| Remaining ~24M params | ~24,000,000 | Int8 + zlib | ~15 MB |
| `train_gpt.py` code | — | raw text | tiny |
| **Total** | | | **just under 16 MB** |

Baseline score: 1.2244 bpb.

## Embeddings at Higher Precision

Several leaderboard entries specifically keep the embedding at FP16 (2 bytes per number) while quantizing everything else harder. Why special treatment?

The embedding is the model's entire vocabulary knowledge — 1024 tokens × 512 dimensions = 524,288 numbers. With tied embeddings, that exact same matrix is also the output layer. It's doing double duty: converting token IDs into 512-dim vectors on the way in, and converting 512-dim vectors back into vocabulary predictions on the way out.

Quantizing it too aggressively hurts disproportionately because errors corrupt both ends of the pipeline. A small rounding error in the embedding for "cat" means "cat" starts with a slightly wrong representation AND the model's ability to predict "cat" as an output gets slightly worse. Those errors compound through the whole model.

At FP16 the embedding costs 524,288 × 2 bytes = ~1MB. That's a meaningful chunk of the 16MB budget, but the quality payoff is worth it.

## The Budget Tradeoff

You have 16,000,000 bytes. The fundamental question is: how many parameters can you afford, and how many bits does each one get?

More parameters with fewer bits each → bigger model (more layers, wider MLP), but less precise weights. Each individual weight is a rougher approximation of what it should be.

Fewer parameters with more bits each → smaller model, but each weight is more precise. Every number is closer to what training intended.

Neither extreme wins. A massive model with 1-bit weights is useless — you've essentially binarized all the knowledge away. A tiny model with perfect 32-bit weights doesn't have enough capacity to learn anything interesting. The competition is about finding the point in between where total intelligence per byte is maximized.

## What Competitors Are Doing

Looking at the leaderboard, the top entries are pushing on three axes at once:

**More layers.** Top entries use 10-11 blocks instead of the baseline's 9. Each additional layer gives the model another round of attention + MLP — another chance to build deeper representations. But each layer adds ~1.8M parameters (the attention projections + MLP matrices), so you need to make room somewhere.

**More aggressive quantization.** Int6 or int5 instead of int8 — fewer bits per number means more rounding error per weight, but you can fit more total parameters in the budget. Going from int8 to int6 saves 25% per parameter. Across 25 million parameters, that's enough room for a couple extra layers or a wider MLP.

**Wider MLPs.** 3× expansion instead of 2× (1536 hidden instead of 1024). This gives each layer more processing capacity — more neurons in the expanded space means more complex nonlinear transformations per block. The MLP matrices are the biggest single cost per block (two matrices of 512×hidden each), so widening them is expensive.

These all trade off against each other within the fixed budget. More layers means more parameters means you need harder quantization to fit, which means each parameter is less precise. A wider MLP means fewer layers or more quantization. Finding the right balance is the game — and the leaderboard shows that different competitors make different bets on where the sweet spot is.

## Quantization-Aware Training (QAT)

The baseline trains in full precision (FP32/BF16) and quantizes afterward. This means the model never sees the rounding errors that quantization will introduce. It trains assuming every weight is a perfect float, then at export time all those weights get rounded to integers. The model had no chance to adapt to that rounding — it's like practicing free throws with a regulation ball and then being handed a slightly different ball for the game.

QAT simulates quantization **during** training. On every forward pass, the weights get fake-quantized: rounded to int8 (or int6 or whatever) and then converted back to float. The model sees the rounding errors as it learns. Gradients flow through the fake quantization (using a straight-through estimator — the gradient pretends the rounding didn't happen, which works well enough in practice).

Over time, the model develops weights that are robust to rounding. Weights naturally drift toward values that land cleanly on integer grid points. The model learns to avoid configurations where rounding one weight cascades into large output errors.

Most top leaderboard entries use QAT. It's one of the bigger free wins available: same model architecture, same parameter count, noticeably better score after compression. All you changed is making training aware of the constraint it'll face at export time.